In [21]:
import os, subprocess, sys
#local imports
pymipl_path = os.path.abspath('../')
sys.path.append(pymipl_path)
sys.path.append( os.path.abspath(pymipl_path+'/xnat_workflow') ) 
from dicom_sort import *
from pathlib import Path


In [22]:
import csv
#read the scan type mappings.
#with open('/workspace/mmilchenko/BM_WU/washu_index_four_mod.csv') as f:
#with open('/workspace/mmilchenko/BM_WU/BM_WU-heuristic-classification-02.26.2026.csv') as f:
#with open('/workspace/mmilchenko/BM_WU/BM_WU-heuristic-classification-02.26.2026-failed.03.03.2026.csv') as f:
#with open('/workspace/mmilchenko/BM_WU/BM_WU-heuristic-classification-02.26.2026-failed.03.10.2026.csv') as f:
with open('/workspace/mmilchenko/BM_WU/BM_WU-heuristic-classification-02.26.2026-rerun.03.31.2026.csv') as f:
    sessions=list(csv.DictReader(f))
num_sessions=len(sessions)

In [23]:
num_sessions

436

In [24]:
sessions[0]

{'subject': 'M17334902',
 'experiment': 'M17334902_20190614173923_del',
 'session': 'TAP02_E08293',
 'T1nc': '2',
 'T1nc_PixelSpacing': '0.972222\\0.972222',
 'T1nc_SeriesDescription': 'SAG_T1_MPRAGE ',
 'T1nc_SliceThickness': '1',
 'T1ce': '31',
 'T1ce_PixelSpacing': '1.09375\\1.09375',
 'T1ce_SeriesDescription': 'T1POST ',
 'T1ce_SliceThickness': '1',
 'T2': '24',
 'T2_PixelSpacing': '1.09375\\1.09375',
 'T2_SeriesDescription': 'TRA_T2_TSE_STEALTH ',
 'T2_SliceThickness': '2',
 'FLAIR': '19',
 'FLAIR_PixelSpacing': '0.71875\\0.71875',
 'FLAIR_SeriesDescription': 'TRA_FLAIR_NEW ',
 'FLAIR_SliceThickness': '5',
 'Error_t1nc': 'Conversion to NII Ok',
 'Error_t1ce': 'Conversion to NII Ok',
 'Error_t2': 'ConversionValidationError',
 'Error_flair': 'Conversion to NII Ok'}

In [33]:
#define global variables and helper functions. 
import importlib
import workflow_adapters as wa
importlib.reload(wa)
import pyxnat
import json
import requests

import logging
import datetime
import yaml

def set_logger():
    root = logging.getLogger()
    root.setLevel(logging.DEBUG)
    
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.DEBUG)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    root.addHandler(handler)

#helper functions
def paths_to_str(x):
    if isinstance(x, Path): return str(x)
    if isinstance(x, dict): return {k: paths_to_str(v) for k, v in x.items()}
    if isinstance(x, list): return [paths_to_str(v) for v in x]
    return x

def resource_to_xnat(local_resource, xnat_session_resource, xnat_project, xnat_subject, xnat_experiment, xnat_interface):
    return wa.sync_resource_xnat(local_resource, xnat_session_resource, xnat_project, xnat_subject, xnat_experiment, \
        level="experiment", create_hierarchy=True,xnat_interface=xnat_interface)

def get_xnat_interface(project):
    host,user,passw,=os.environ.get('XNAT_HOST'),os.environ.get('XNAT_USER'),os.environ.get('XNAT_PASS')
    xnat = pyxnat.Interface(server=host, user=user, password=passw)
    xnat.select.project(project)
    return xnat

def launch_cs_command(xnat_interface,project,subject,experiment,session,workflow_id,xnat_command_id,xnat_wrapper_name,verbose=False):
    host=os.environ.get('XNAT_HOST')
    url = f"{host}/xapi/projects/{project}/commands/{xnat_command_id}/wrappers/{xnat_wrapper_name}/launch"
    try:
        body = {
            "PROJECT": project,
            "SUBJECT": subject,
            "EXPERIMENT": experiment,
            "WORKFLOW_ID": workflow_id,
            "MICROENV": "NONE",
            "REPO_GIT": "NONE", 
            "session": session            
        }
        resp = xnat_interface._http.post(url, json=body)
        if verbose:
            print("Request body: ", body)
            print("Request url: ", url)
            print("Status:", resp.status_code)
            print("Reason:", resp.reason)
            print("URL:", resp.url)
            print("Headers:\n", resp.headers)
            print("Cookies:\n", resp.cookies)
            print("Elapsed:", resp.elapsed)
            print("Text:\n", resp.text)
        return resp.status_code in (200, 201, 202)
    except Exception as e:
        print(e)
        return False
        
global_vars={}
env_type='jupyter'
#env_type='container'
project='BM_WU'
workflow_id='BRATS_preproc_v2'
xnat_command_id=22
xnat_wrapper_id='xnat-ai-workflow'#40

global_vars['g_workflow_id']=workflow_id
root_dir=Path('/workspace/mmilchenko')
local_workdir_path=Path('/workspace/mmilchenko/BM_WU')

if env_type=='jupyter':

    #path that mounts directory with XNAT experiments
    global_vars['g_input_mount_path']=Path('/data/projects/BM_WU/experiments')
    #path to local directory where processing will be stored
    global_vars['g_local_workdir_path']=Path('/workspace/mmilchenko/BM_WU')
    #library locations, algorithm specific
    global_vars['g_pymipl_dir']=root_dir / "pymipl"
    #main algorithm repository dir
    global_vars['g_env_repo_dir']=root_dir / 'envs/brats'
    #global_vars['g_alg_repo_dir']
    global_vars['g_project']=project
        
elif env_type == 'container': #built-in defaults used in the bootstrap image.
    wa.init_global_vars_bootstrap_image(global_vars,project)



In [34]:
!pip install dicom2nifti
import dicom2nifti

def test_dcm2nii_error(session,global_vars):
    rt = global_vars['g_input_mount_path'] / session['experiment'] / 'SCANS'
    
    t1nc = rt / session['T1nc'] / 'DICOM'
    t1ce = rt / session['T1ce'] / 'DICOM'
    t2 = rt / session['T2'] / 'DICOM'
    flair = rt / session ['FLAIR'] / 'DICOM'    

    names=('t1nc','t1ce','t2','flair')
    
    out_file=global_vars['g_local_workdir_path'] / 'temp/temp_out.nii'
    for scan,name in zip((t1nc,t1ce,t2,flair),names):
        try:
            dicom2nifti.dicom_series_to_nifti(scan, out_file, reorient_nifti=True)
            session['Error_'+name]='Conversion to NII Ok'
        except Exception as e:
            if isinstance(e, dicom2nifti.exceptions.ConversionValidationError):
                session['Error_'+name]='ConversionValidationError'
            else: 
                session['Error_'+name]=str(e)
        


In [35]:
#loop over sessions to create batch scripts.
#! pip install nibabel

import pyxnat

dt=datetime.datetime.now().strftime("%Y%m%d_%H%M")
batch_file=local_workdir_path / f"batch_{dt}.sh"
#n indicates starting position in the spreadsheet.
n=0

xnat_interface=None
start_pos=4
end_pos=500
subjects={}
nExp=0
nFailed=0
sessions_failed=[]

for session in sessions:
    
    if Path('/data/projects/BM_WU/experiments/'+session['experiment']+'/RESOURCES/BRATS_preproc_v2/bet/'+session['experiment'] +'_t1_brain.nii.gz').exists():
        subjects[session['subject']]=1
        nExp=nExp+1
        if nExp % 100 == 0:
            print(f"found {nExp} experiments")
    else:
        nFailed=nFailed+1
        if nFailed % 10 == 0: print(f"Found failed: {nFailed}")
        test_dcm2nii_error(session,global_vars) 
        sessions_failed+=[session]
        #if nFailed % 10 == 20:
        #    break #DEBUG
    continue
    '''
    is_corrupt=False
    names=('t1nc','t1ce','t2','flair')
    for name in names: 
        if session[ 'Error_'+ name ] == 'ConversionValidationError':
            is_corrupt=True
            break
    if not is_corrupt:
        print(f'Checked {session["experiment"]} for corruption, proceed' )
    else: 
        continue
    '''
    n=n+1    
    if n<start_pos or n>end_pos:
        continue
    
    job,steps={},[]
    job_experiment=session['experiment']
    job_scan_context=global_vars['g_input_mount_path'] #/ job_scan_id / 'DICOM'
    if env_type == 'jupyter': job_scan_context = job_scan_context / Path(job_experiment)
    job_scan_context = job_scan_context / Path('SCANS')    
    
    job['job_title']=f"Workflow {workflow_id}, subject {session['subject']}, experiment {session['experiment']}"
    job['job_t1w_path'] = job_scan_context / session['T1nc'] / 'DICOM'
    job['job_t1wce_path'] = job_scan_context / session['T1ce'] / 'DICOM'
    job['job_t2w_path'] = job_scan_context / session['T2'] / 'DICOM'
    job['job_t2f_path'] = job_scan_context / session['FLAIR'] / 'DICOM'
    job['job_subject'] = session['subject']
    job['job_exp_label'] =session['experiment']
    job_id=f"{workflow_id}_{session['subject']}_{session['experiment']}"
    job['job_id']=job_id
    
    #directory where the job writes local files. This line must work for both types of workflow.
    #TODO. This is not right, because global_vars['g_local_workdir_path'] may points to 
    # container workdir but it cannot be used inside jupyter. 
    # so we'd need class or struct or other type of variable scope separation to separate 'this' script variables from 'job' or 'step' variables that are used =
    # to generate workflows. In case of jypyter, local scope and script scopes are the same;
    # but in case of jupyter generating variables to run in container, these would be different. 
    
    job['job_workdir']=global_vars['g_local_workdir_path'] / session['subject'] / session['experiment']
         
    #Step 1. Run BRATS preprocessing. 
    step={"step_title": "1. Run BRATS preprocessing"}
    step['step_command']="micromamba run -p {g_env_repo_dir} python {g_pymipl_dir}/brats_preprocess.py --patient_id {job_exp_label} \
        --outdir {job_workdir} --t1 {job_t1w_path} --t1ce {job_t1wce_path} --t2 {job_t2w_path} --flair {job_t2f_path}"
    steps+=[step]

    #Steps 2-5. Generate QC images.    
    step={"step_title": "2. T1w QC image"}
    step['step_command']="micromamba run -p {g_env_repo_dir} python {g_pymipl_dir}/slice_qc.py \
        --mask {job_workdir}/bet/brain_mask.nii.gz -o {job_workdir}/qc/t1_qc.png {job_workdir}/raw/t1.nii.gz"
    steps+=[step]

    step={"step_title": "3. T1ce QC image"}
    step['step_command']="micromamba run -p {g_env_repo_dir} python {g_pymipl_dir}/slice_qc.py \
        --mask {job_workdir}/bet/brain_mask.nii.gz -o {job_workdir}/qc/t1ce_qc.png {job_workdir}/raw/t1ce.nii.gz"
    steps+=[step]
    
    step={"step_title": "4. T2w QC image"}
    step['step_command']="micromamba run -p {g_env_repo_dir} python {g_pymipl_dir}/slice_qc.py \
        --mask {job_workdir}/bet/brain_mask.nii.gz -o {job_workdir}/qc/t2_qc.png {job_workdir}/raw/t2.nii.gz"
    steps+=[step]

    step={"step_title": "5. FLAIR QC image"}
    step['step_command']="micromamba run -p {g_env_repo_dir} python {g_pymipl_dir}/slice_qc.py \
        --mask {job_workdir}/bet/brain_mask.nii.gz -o {job_workdir}/qc/flair_qc.png {job_workdir}/raw/flair.nii.gz"
    steps+=[step]

    #Step 6. Cleanup.
    step={ "step_title": "6. Cleanup symlinks" }
    step['step_command']="rm -f {job_workdir}/bet/brain_mask.nii.gz"
    steps+=[step]

    #Steps 7-10. Rename images for storage.
#    print(job['job_exp_label'])
#    step={"step_title": "Rename output T1nc image"}
#    step['step_command']="mv {job_workdir}/bet/t1.nii.gz {job_workdir}/bet/{job_exp_label}_t1_aligned.nii.gz"
#    steps+=[step]

#    step={"step_title": "Rename output T1ce image"}
#    step['step_command']="mv {job_workdir}/bet/t1ce.nii.gz {job_workdir}/bet/{job_exp_label}_t1ce_aligned.nii.gz"
#    steps+=[step]

#    step={"step_title": "Rename output T2w image"}
#    step['step_command']="mv {job_workdir}/bet/t2.nii.gz {job_workdir}/bet/{job_exp_label}_t2_aligned.nii.gz"
#    steps+=[step]

#    step={"step_title": "Rename output FLAIR image"}
#    step['step_command']="mv {job_workdir}/bet/flair.nii.gz {job_workdir}/bet/{job_exp_label}_flair_aligned.nii.gz"
#    steps+=[step]

    #Step 7. Upload results to session resource. Test for notebook mode only. Container service mode does this internally.
    step={ "step_title": "Upload results" }
    if env_type=='jupyter':
        step['step_command']='micromamba run -p {g_env_repo_dir} python {g_pymipl_dir}/xnat_workflow/sync-resource-with-xnat.py \
            --level experiment --project {g_project} --subject {job_subject} --experiment {job_exp_label} --local_resource {job_workdir} \
            --remote_resource {g_workflow_id} --create_hierarchy 1'        
    else:
        step['step_command']='micromamba run -n base python {g_pymipl_dir}/xnat_workflow/sync-resource-with-xnat.py \
            --level experiment --project {g_project} --subject {job_subject} --experiment {job_exp_label} --local_resource {job_workdir} \
            --remote_resource {g_workflow_id} --create_hierarchy 1'                
    steps+=[step]
    
    #process job and write out
    job['steps']=steps
    job_id=f"{workflow_id}_{session['subject']}_{job_experiment}"
    local_job_dir=local_workdir_path / 'jobs' / job_id
    job_file_yaml=local_job_dir / 'job.yaml'
    job_file_sh=local_job_dir / 'job.sh'
    local_job_dir.mkdir(parents=True,exist_ok=True)
    
    with open(job_file_yaml,"w") as f:
        yaml.safe_dump(paths_to_str(job),f,sort_keys=False)

    if env_type=='jupyter': #write all commands to a single batch file
        wa.workflow_to_batch(job,global_vars,batch_file)
        print(batch_file)
        break #DEBUG
        
    else: #one batch file per job
        #create batch        
        print(job_file_sh)
        #reset job script
        ! truncate -s 0 {job_file_sh}
        #generate job script
        #DEBUG
        wa.workflow_to_batch(job,global_vars,job_file_sh)
        ! chmod +x {job_file_sh}
        #store batch file.
        print('sending batch to xnat resource')
        #break #DEBUG
        #res=0
        #TODO this call is indicative of the mixture of scopes issue described above.
        if xnat_interface is None: xnat_interface=get_xnat_interface(project)
        res1=resource_to_xnat(local_job_dir, workflow_id, project, job['job_subject'], job['job_exp_label'],xnat_interface)
        print('submitting job to Container Service')
        #print(f"launching CS command for {project}, {job['job_subject']} ,{job['job_exp_label']}, {workflow_id}, {xnat_command_wrapper_id}") 
        #res2=launch_cs_command(xnat_interface,project,job['job_subject'],job['job_exp_label'],session['session'],workflow_id,15,"xnat-ai-workflow",verbose=True)
        #res2=launch_cs_command(xnat_interface,project,job['job_subject'],job['job_exp_label'],session['session'],workflow_id,15,26,verbose=True)
        exp_id=xnat_interface.select.project(project).subject(job['job_subject']).experiment(job['job_exp_label']).id()
        res2=launch_cs_command(xnat_interface,project,job['job_subject'],job['job_exp_label'],exp_id,workflow_id,xnat_command_id,xnat_wrapper_id,verbose=True)
        #res=launch_cs_command(xnat_interface,"BM_WU","M00101754","M00101754_20180719144757","TAP02_E01992",workflow_id,15,"xnat-ai-workflow")     
        
        if res1 == 0 and res2:
            if n % 10 == 0: print(f'Done {n} out of {num_sessions} ({n*100/num_sessions:.1f}%)')
        else: 
            print (f'Failed sending configuration to session resource, error status 1 {res1}, error status 2 {res2}. Stopping execution.')
            break
    #break #DEBUG

if xnat_interface is not None: xnat_interface.disconnect()
        
#if n>1: break
#break #debug: do just one run.

Found failed: 10


Found failed: 20


found 100 experiments


Found failed: 30


Found failed: 40


...

Found failed: 60


Found failed: 70
Found failed: 80


Found failed: 90


Found failed: 100


Found failed: 110
Found failed: 120
Found failed: 130
Found failed: 140
Found failed: 150
Found failed: 160
Found failed: 170
Found failed: 180


In [31]:
len(sessions_failed)

180

In [18]:
n

51

In [9]:
len(sessions_failed)

487

In [36]:
with open('failed_sessions.04.01.2026.csv','w',newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, sessions_failed[0].keys())
    dict_writer.writeheader()
    dict_writer.writerows(sessions_failed)

In [ ]:
! pip install dicom2nifti
import dicom2nifti
dcm="/data/projects/BM_WU/experiments/M99756740_20250607155048/SCANS/5/DICOM"
out_file='/workspace/mmilchenko/BM_WU/scan5.nii'
dicom2nifti.dicom_series_to_nifti(dcm, out_file, reorient_nifti=True)